# Bantay-Baha: raw data check and quick EDA

One pass over every dataset in the concept proposal (Appendix B, Table 1): does it exist, does it download, does it cover
what the proposal assumes? Data come from `python scripts/download_data.py` (open sources) and
`python scripts/download_gee.py --project <id>` (Earth Engine sources). Sections whose files are missing print `PENDING` and skip.

| # | Layer | Dataset | Folder |
|---|---|---|---|
| 1 | Target | DOH-Epi dengue, province/city x week, 2016 to Jan 2021 (HDX) | `dengue_hdx` |
| 2 | Target (lepto) | Project CCHAIN PIDSR city-week disease totals, 2008-2022 | `cchain` |
| 3 | Cross-check | Mendeley Parreño 2023 national weekly notifiable diseases, 2019-2023 | `mendeley_parreno` |
| 4 | Cross-check | OpenDengue V1.3 | `opendengue` |
| 5 | Join keys | HDX COD-AB boundaries; 2020 CPH population | `cod_ab`, `census2020` |
| 6 | Storms | IBTrACS v04r01 WP | `ibtracs` |
| 7 | Rain | CHIRPS v2.0 daily 0.05 deg; GPM IMERG V07 (GEE) | `chirps`, `imerg` |
| 8 | Flood | Copernicus EMS EMSR312/EMSR478; UP NOAH 5/25-yr hazard; GFD + Sentinel-1 inventory (GEE) | `copernicus_ems`, `noah`, `gfd`, `sentinel1` |
| 9 | Rivers | HydroRIVERS v1.0 Asia | `hydrorivers` |
| 10 | Access | OSM (Geofabrik), roroph RoRo routes, HOT OSM + NOAH health facilities, MAP friction 2019 (GEE) | `osm`, `roroph`, `hotosm_health`, `noah_facilities`, `map_friction` |
| 11 | Population | WorldPop 2020 100 m; Meta HRSL under-5 / 60+ | `worldpop`, `meta_hrsl` |

In [ ]:
import re, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .3})

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
BBOX = (116, 4, 127, 21.5)
FINDINGS = []  # (dataset, finding) collected for the summary at the end
def note(ds, msg): FINDINGS.append((ds, msg)); print(f"[{ds}] {msg}")
def have(*parts):
    p = RAW.joinpath(*parts)
    ok = p.exists() and (p.is_file() or any(p.iterdir()))
    if not ok: print(f"PENDING: {p.relative_to(ROOT)} not downloaded")
    return ok

if (RAW / "MANIFEST.csv").exists():
    man = pd.read_csv(RAW / "MANIFEST.csv")
else:  # fall back to what is on disk
    fs = [p for p in RAW.rglob("*") if p.is_file() and p.suffix != ".part" and p.name != "MANIFEST.csv"]
    man = pd.DataFrame({"source": [p.relative_to(RAW).parts[0] for p in fs], "path": fs, "bytes": [p.stat().st_size for p in fs]})
man.groupby("source").agg(files=("path", "size"), MB=("bytes", lambda b: round(b.sum() / 1e6, 1))).sort_values("MB", ascending=False)

## 1. DOH-Epi dengue (HDX), the forecast target

In [ ]:
dg = pd.read_csv(RAW / "dengue_hdx/doh-epi-dengue-data-2016-2021.csv", skiprows=[1])  # row 1 = HXL tags
dg["loc"] = dg["loc"].str.strip(); dg["date"] = pd.to_datetime(dg["date"])
print(dg.shape, "| weeks:", dg.date.nunique(), dg.date.min().date(), "->", dg.date.max().date(), "| weekday:", dg.date.dt.day_name().unique())
print("locations:", dg["loc"].nunique(), "| regions:", dg.Region.nunique())
cities = sorted(l for l in dg["loc"].unique() if "CITY" in l or l in ("PATEROS", "TACLOBAN"))
print(f"{len(cities)} city-level units:", cities)
yr = dg.groupby(dg.date.dt.year).agg(rows=("cases", "size"), cases=("cases", "sum"), deaths=("deaths", "sum"), units=("loc", "nunique"))
yr["CFR_%"] = (yr.deaths / yr.cases * 100).round(2); yr

In [ ]:
full = pd.MultiIndex.from_product([dg["loc"].unique(), np.sort(dg.date.unique())])
miss = 1 - dg.set_index(["loc", "date"]).reindex(full).cases.notna().groupby(level=1).mean()
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
dg.groupby("date").cases.sum().plot(ax=ax[0], color="#1f5f99"); ax[0].set_ylabel("cases / week (national)")
miss.plot(ax=ax[1], color="#b5482b"); ax[1].set_ylabel("share of units missing")
for d, n in [("2018-09-15", "Ompong"), ("2019-12-02", "Tisoy"), ("2020-11-01", "Rolly"), ("2020-11-11", "Ulysses")]:
    for a in ax: a.axvline(pd.Timestamp(d), color="grey", ls=":", lw=1)
    ax[0].text(pd.Timestamp(d), ax[0].get_ylim()[1] * .9, n, rotation=90, fontsize=8, va="top")
plt.tight_layout(); plt.show()
print("duplicate loc-week rows:", dg.duplicated(["loc", "date"]).sum(), "| negative cases:", (dg.cases < 0).sum(), "| units with 0 cases all period:", (dg.groupby("loc").cases.sum() == 0).sum())
note("dengue_hdx", f"{dg['loc'].nunique()} reporting units (provinces + {len(cities)} HUC/NCR cities), not 81 provinces; HUCs must be folded into their geographic province (or kept as separate nodes).")
note("dengue_hdx", f"2016 deaths = {yr.loc[2016,'deaths']:.0f} (CFR {yr.loc[2016,'CFR_%']}%), implausibly high vs other years; treat deaths as unreliable.")
note("dengue_hdx", f"2020 cases = {yr.loc[2020,'cases']:,} vs 2019 {yr.loc[2019,'cases']:,} (COVID reporting collapse), consistent with proposal using 2020 for access only. Data end {dg.date.max().date()}.")

## 2. Project CCHAIN PIDSR city-week totals (leptospirosis stretch goal)

In [ ]:
cc = pd.read_csv(RAW / "cchain/disease_pidsr_totals.csv", parse_dates=["date"])
loc = pd.read_csv(RAW / "cchain/location.csv").drop_duplicates("adm3_pcode")[["adm3_pcode", "adm3_en", "adm2_en"]]
cc = cc.merge(loc, on="adm3_pcode", how="left")
print(cc.shape, cc.date.min().date(), "->", cc.date.max().date(), "| cities:", cc.adm3_en.nunique())
print(sorted(cc.adm3_en.dropna().unique()))
cc.pivot_table(index="disease_common_name", columns=cc.date.dt.year, values="case_total", aggfunc="sum").loc[["DENGUE FEVER", "LEPTOSPIROSIS"]]

In [ ]:
lep = cc[cc.disease_common_name == "LEPTOSPIROSIS"].pivot_table(index="date", columns="adm3_en", values="case_total", aggfunc="sum")
ax = lep.loc["2016":"2021"].plot(figsize=(11, 3.5), lw=.9, legend=False, title="Leptospirosis, cases / week, CCHAIN cities 2016-2021")
for d in ["2018-09-15", "2019-12-02", "2020-11-01", "2020-11-11"]: ax.axvline(pd.Timestamp(d), color="grey", ls=":")
ax.legend(fontsize=7, ncol=4, loc="upper left"); plt.show()
note("cchain", f"Open city-week LEPTOSPIROSIS series exists (PIDSR via CCHAIN, {cc.adm3_en.nunique()} cities, 2008-2022). The proposal's 'no open subnational lepto series' is outdated; lepto can be a sub-national stretch goal without digitising DOH reports.")

## 3. Mendeley Parreño national weekly notifiable diseases

In [ ]:
mx = pd.read_excel(RAW / "mendeley_parreno/Diseases_Data_for_NBL.xlsx")
long = mx.melt(id_vars="Week", var_name="col", value_name="cases")
long[["disease", "year"]] = long.col.str.rsplit(" - ", n=1, expand=True); long.year = long.year.astype(int)
print(mx.shape, "| diseases:", long.disease.nunique(), "| years:", sorted(long.year.unique()))
cov = long.assign(present=long.cases.notna()).pivot_table(index="disease", columns="year", values="present", aggfunc="sum")
display(cov.loc[["Dengue", "Leptospirosis"]])
sub = long[long.disease.isin(["Dengue", "Leptospirosis"])].pivot_table(index=["year", "Week"], columns="disease", values="cases")
sub.plot(subplots=True, figsize=(11, 4), title="National weekly (Mendeley)", legend=True); plt.show()
note("mendeley_parreno", f"National weekly only, years {sorted(long.year.unique())}; 2020 weeks present: {int(cov.loc['Dengue', 2020])}/52. Does not cover 2016-2018, so it cannot cross-check Ompong (2018).")

## 4. OpenDengue V1.3 (Philippines)

In [ ]:
z = zipfile.ZipFile(RAW / "opendengue/Temporal_extract_V1_3.zip")
od = pd.read_csv(z.open([n for n in z.namelist() if n.endswith(".csv")][0]), low_memory=False)
od = od[od.adm_0_name.str.upper() == "PHILIPPINES"].assign(start=lambda d: pd.to_datetime(d.calendar_start_date))
display(od.groupby(["S_res", "T_res"]).agg(rows=("dengue_total", "size"), first=("Year", "min"), last=("Year", "max")))
nat = od[(od.S_res == "Admin0") & (od.T_res == "Week")].set_index("start").dengue_total.sort_index()
hdx_nat = dg.groupby("date").cases.sum()
j = pd.concat([nat.rename("opendengue"), hdx_nat.rename("doh_hdx")], axis=1).loc["2016":"2020"].dropna()
print(f"overlapping weeks 2016-2020: {len(j)}; correlation {j.corr().iloc[0,1]:.3f}; ratio OD/HDX median {(j.opendengue/j.doh_hdx).median():.2f}")
note("opendengue", "V1.3 exists (May 2025). PHL = national weekly 2012-2023, region yearly to 2021, province monthly only 1994-2010; useful only as a national cross-check.")

## 5. Boundaries (COD-AB) and 2020 census: join-key check

In [ ]:
import pyogrio
zpath = f"zip://{(RAW / 'cod_ab/phl_admin_boundaries.gdb.zip').as_posix()}"
print(pyogrio.list_layers(zpath))
adm1 = gpd.read_file(zpath, layer="phl_admin1")
adm2 = gpd.read_file(zpath, layer="phl_admin2")
adm3 = gpd.read_file(zpath, layer="phl_admin3", read_geometry=False)
print("ADM1", len(adm1), "| ADM2", len(adm2), "| ADM3", len(adm3), "| version:", adm2.version.iloc[0], "valid_on:", adm2.valid_on.iloc[0])
print(sorted(adm2.adm2_name))

In [ ]:
def norm(s):
    s = re.sub(r"\(.*?\)", "", str(s).upper())
    return re.sub(r"[^A-Z]", "", s.replace("CITY OF ", "").replace(" CITY", ""))
# dengue spellings / old units -> COD-AB ADM2 (by normalised name); Maguindanao was split in 2022
fix = {"NUEVAVISCAYA": "NUEVAVIZCAYA", "SISQUIJOR": "SIQUIJOR", "NORTHCOTABATO": "COTABATO", "BASILIG": "BASILAN",
       "ISABELA": "ISABELA", "MAGUINDANAO": "MAGUINDANAODELNORTE", "MANILA": "METROPOLITANMANILAFIRSTDISTRICT"}
a2 = {norm(n): n for n in adm2.adm2_name}
a3 = adm3.assign(k=adm3.adm3_name.map(norm)).drop_duplicates("k").set_index("k")
rows = []
for l in sorted(dg["loc"].unique()):
    k = fix.get(norm(l), norm(l))
    if k in a2: rows.append((l, "ADM2", a2[k]))
    elif k in a3.index: rows.append((l, "ADM3 city -> ADM2", a3.loc[k, "adm2_name"]))
    else: rows.append((l, "UNMATCHED", None))
m = pd.DataFrame(rows, columns=["dengue_loc", "level", "adm2"])
print(m.level.value_counts().to_dict()); display(m[m.level == "UNMATCHED"])
unused = sorted(set(adm2.adm2_name) - set(m.adm2.dropna()))
print(f"COD-AB ADM2 units with no dengue unit mapped ({len(unused)}):", unused)
note("cod_ab", f"COD-AB is current-boundary ({len(adm2)} ADM2: 4 NCR districts, Maguindanao del Norte/Sur split 2022, Special Geographic Area 2024). The 2016-2020 dengue units need a harmonised historical crosswalk. Name match: {(m.level!='UNMATCHED').sum()}/{len(m)} units. PSA PSGC 2Q 2026 must be downloaded manually (psa.gov.ph blocks scripts).")

In [ ]:
if have("census2020"):
    cx = pd.ExcelFile(RAW / "census2020/2020_CPH_SexAge_Popn_Mun_Admin3.xlsx"); print(cx.sheet_names)
    cph = pd.read_excel(cx, cx.sheet_names[0]); print(cph.shape); display(cph.head(3))

## 6. IBTrACS typhoon tracks (backtest storms)

In [ ]:
ib = pd.read_csv(RAW / "ibtracs/ibtracs.WP.list.v04r01.csv", skiprows=[1], low_memory=False, na_values=[" "])
STORMS = {("MANGKHUT", 2018): "Ompong", ("KAMMURI", 2019): "Tisoy", ("GONI", 2020): "Rolly", ("VAMCO", 2020): "Ulysses"}
tr = ib[[(n, s) in STORMS for n, s in zip(ib.NAME, ib.SEASON)]].copy()
tr["local"] = tr.apply(lambda r: STORMS[(r.NAME, r.SEASON)], axis=1); tr["ISO_TIME"] = pd.to_datetime(tr.ISO_TIME)
r34 = ["USA_R34_NE", "USA_R34_SE", "USA_R34_SW", "USA_R34_NW"]
s = tr.groupby("local").agg(sid=("SID", "first"), fixes=("ISO_TIME", "size"), start=("ISO_TIME", "min"), end=("ISO_TIME", "max"),
                            max_wind_kt=("USA_WIND", "max"), r34_coverage=("USA_R34_NE", lambda x: x.notna().mean().round(2)))
display(s)
fig, ax = plt.subplots(figsize=(6, 7)); adm1.boundary.plot(ax=ax, lw=.3, color="grey")
for n, g in tr.groupby("local"): ax.plot(g.LON, g.LAT, lw=1.5, label=n)
ax.set(xlim=(113, 132), ylim=(4, 22), title="Backtest typhoon tracks"); ax.legend(); plt.show()
print("WP storms per season 2016-2020:", ib[ib.SEASON.between(2016, 2020)].groupby("SEASON").SID.nunique().to_dict())
note("ibtracs", f"All 4 storms present; USA 34-kt radius available on {s.r34_coverage.min():.0%}-{s.r34_coverage.max():.0%} of fixes, so the gale-radius buffer needs a fallback (e.g. TOKYO_R30 or a fixed radius).")

## 7. Rainfall: CHIRPS (open) and IMERG (GEE)

In [ ]:
import xarray as xr
if have("chirps"):
    ch = xr.open_mfdataset(sorted((RAW / "chirps").glob("*.nc")), decode_times=False, combine="by_coords")
    print(ch)
    var = list(ch.data_vars)[0]; tname = [d for d in ch[var].dims if d.upper().startswith("T")][0]
    ch = xr.decode_cf(ch.assign_coords({tname: ch[tname]}), decode_times=True) if "since" in str(ch[tname].attrs.get("units", "")) else ch
    pr = ch[var]
    daily = pr.mean([d for d in pr.dims if d != tname]).to_series(); print("days:", len(daily), "| NaN share:", float(pr.isnull().mean()))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
    daily.plot(ax=ax[0], lw=.5, color="#1f5f99", title="CHIRPS daily mean rainfall over PHL bbox (mm/day)")
    (pr.sum(tname) / 5).plot(ax=ax[1], cmap="Blues", robust=True); ax[1].set_title("mean annual total (mm)")
    plt.tight_layout(); plt.show()
if have("imerg"):
    import rasterio
    fs = sorted((RAW / "imerg").glob("*.tif")); print(len(fs), "monthly IMERG stacks")

## 8. Flood: Copernicus EMS, NOAH hazard, GFD / Sentinel-1 (GEE)

In [ ]:
def read_zip_layers(z, pattern):
    out = []
    for n in zipfile.ZipFile(z).namelist():
        if re.search(pattern, n, re.I) and n.endswith((".shp", ".json", ".geojson")):
            g = gpd.read_file(f"zip://{Path(z).as_posix()}!{n}"); g["layer"] = Path(n).stem; g["file"] = Path(z).name; out.append(g)
    return out
for act in ["EMSR312", "EMSR478"]:
    zs = sorted((RAW / "copernicus_ems" / act).glob("*.zip")); print(f"{act}: {len(zs)} vector packages")
    for z in zs[:1]: print("  layers in", z.name, ":", [Path(n).stem for n in zipfile.ZipFile(z).namelist() if n.endswith((".shp", ".json"))])
obs = [g for z in sorted((RAW / "copernicus_ems/EMSR312").glob("*DELINEATION*.zip")) for g in read_zip_layers(z, r"observed.?event")]
if obs:
    fl = pd.concat([g.to_crs(32651) for g in obs]); fl["km2"] = fl.area / 1e6
    display(fl.groupby("file").km2.sum().round(1))
    note("copernicus_ems", f"EMSR312 (Ompong) observed-flood polygons: {fl.km2.sum():.0f} km2 across {fl.file.nunique()} delineation packages (Northern Luzon AOIs only).")
note("copernicus_ems", "EMSR478 (Rolly) has only damage-GRADING products (4 AOIs), no flood delineation; it cannot cross-check Sentinel-1 flood extent as the proposal states.")

In [ ]:
for rp in ["5yr", "25yr"]:
    if have("noah", rp):
        zs = sorted((RAW / "noah" / rp).glob("*.zip")); print(rp, len(zs), "province files,", round(sum(z.stat().st_size for z in zs) / 1e6), "MB")
z = sorted((RAW / "noah/25yr").glob("Albay*.zip"))[0]
shp = [n for n in zipfile.ZipFile(z).namelist() if n.endswith(".shp")]; print("example:", z.name, shp)
nz = gpd.read_file(f"zip://{z.as_posix()}!{shp[0]}"); print(nz.columns.tolist(), nz.crs)
vcol = [c for c in nz.columns if c.lower() in ("var", "haz", "hazard", "class")][0]
nz = nz.to_crs(32651); display(nz.assign(km2=nz.area / 1e6).groupby(vcol).km2.sum().round(1))
nz.plot(column=vcol, cmap="Blues", figsize=(5, 5), legend=True); plt.title("NOAH 25-yr flood hazard, Albay"); plt.show()
noah_prov = {re.sub(r"\.zip$", "", z.name) for z in (RAW / "noah/25yr").glob("*.zip")}
note("noah", f"NOAH 5-yr: {len(list((RAW/'noah/5yr').glob('*.zip')))} files, 25-yr: {len(noah_prov)} files (per province); coverage is partial, so 'share of population in hazard zone' is undefined for provinces without maps. Source is the UP NOAH Google Drive, not LiPAD.")

In [ ]:
if have("gfd"):
    gfd = pd.read_csv(RAW / "gfd/gfd_events_phl.csv"); print(len(gfd), "GFD events intersecting PHL bbox"); display(gfd.tail())
    note("gfd", "Global Flood Database ends in 2018, so it can cover Ompong at most, not Tisoy/Rolly/Ulysses.")
if have("sentinel1"):
    s1 = pd.read_csv(RAW / "sentinel1/s1_scene_inventory.csv"); s1["t"] = pd.to_datetime(s1.time_ms, unit="ms")
    s1["phase"] = np.where(s1.t < pd.to_datetime(s1.landfall), "before", "after"); display(s1.pivot_table(index="typhoon", columns="phase", values="scene", aggfunc="count"))

## 9. HydroRIVERS (Philippines clip)

In [ ]:
hr = gpd.read_file(f"zip://{(RAW / 'hydrorivers/HydroRIVERS_v10_as_shp.zip').as_posix()}!HydroRIVERS_v10_as_shp/HydroRIVERS_v10_as.shp", bbox=BBOX)
hr = hr.clip(adm1.to_crs(hr.crs).union_all()) if "adm1" in dir() else hr
print(len(hr), "reaches |", hr.columns.tolist())
print("outlets (NEXT_DOWN==0):", (hr.NEXT_DOWN == 0).sum(), "| ORD_STRA dist:", hr.ORD_STRA.value_counts().sort_index().to_dict())
ax = hr.plot(linewidth=hr.ORD_STRA / 4, figsize=(6, 7), color="#1f5f99"); ax.set_title("HydroRIVERS reaches, PHL"); plt.show()

## 10. Access network: OSM, RoRo routes, health facilities, friction

In [ ]:
import rdata
rr = rdata.read_rda(RAW / "roroph/roro_routes.rda")["roro_routes"]; print(rr.shape); display(rr.head())
provs = pd.unique(rr[["from_prov", "to_prov"]].values.ravel()); print("provinces in RoRo network:", len(provs), "| highway_type:", rr.highway_type.value_counts().to_dict())
display(rr[["dist_nm", "avg_time_hrs", "freq_daily"]].describe().round(1))
coarse = ((rr[["from_lat", "from_lon", "to_lat", "to_lon"]] * 10) % 1 == 0).all(axis=1).mean()
fig, ax = plt.subplots(figsize=(6, 7)); adm1.boundary.plot(ax=ax, lw=.3, color="grey")
for _, r in rr.iterrows(): ax.plot([r.from_lon, r.to_lon], [r.from_lat, r.to_lat], color="#1f5f99", lw=.4 + r.freq_daily / 4)
ax.set(xlim=(116, 127), ylim=(4, 21.5), title="roroph RoRo links (width = daily trips)"); plt.show()
note("roroph", f"{len(rr)} routes across {len(provs)} provinces, with GADM-style names (e.g. 'AgusandelNorte'), so a name crosswalk is needed. Port coordinates are coarse ({coarse:.0%} of routes have every coordinate on a 0.1 deg grid) and need snapping to OSM ports. Routes describe 2024-2026 operations, so they are a proxy for 2018-2020.")

In [ ]:
pbf = RAW / "osm/philippines-latest.osm.pbf"
if have("osm", "philippines-latest.osm.pbf"):
    print(f"PBF {pbf.stat().st_size/1e6:.0f} MB; OSM layers:", pyogrio.list_layers(pbf.as_posix()))
    roads = pyogrio.read_dataframe(pbf.as_posix(), layer="lines", columns=["highway", "other_tags"], read_geometry=False)
    print("lines:", len(roads))
    hw = roads.highway.value_counts(); display(hw.head(15))
    ferries = roads.other_tags.fillna("").str.contains('"route"=>"ferry"').sum()
    note("osm", f"Geofabrik PBF has {hw.sum():,} highway ways ({hw[['motorway','trunk','primary','secondary']].sum():,} motorway-secondary) and {ferries} route=ferry lines; it is a current snapshot, not a 2018-2020 one.")

In [ ]:
hz = zipfile.ZipFile(RAW / "hotosm_health/hotosm_phl_health_facilities_osm_gpkg.zip"); gp = [n for n in hz.namelist() if n.endswith(".gpkg")][0]
hf = gpd.read_file(f"zip://{(RAW / 'hotosm_health/hotosm_phl_health_facilities_osm_gpkg.zip').as_posix()}!{gp}")
print("HOT OSM:", hf.shape); display(hf.amenity.value_counts().head(8) if "amenity" in hf else hf.columns)
nf = gpd.read_file(RAW / "noah_facilities/hospitals.geojson"); print("NOAH facilities:", nf.shape, nf.amenity.value_counts().to_dict())
fig, ax = plt.subplots(figsize=(6, 7)); adm1.boundary.plot(ax=ax, lw=.3, color="grey")
hf[hf.get("amenity", "") == "hospital"].plot(ax=ax, markersize=1, color="#b5482b", label="OSM hospital")
ax.set(xlim=(116, 127), ylim=(4, 21.5), title="OSM hospitals"); ax.legend(); plt.show()
note("hospitals", f"DOH NHFR blocks scripted access (403) - must be downloaded manually. Open fallbacks: HOT OSM ({(hf.get('amenity','')=='hospital').sum()} hospitals) and NOAH critical facilities ({len(nf):,} points, name only, no level/bed data).")
if have("map_friction"):
    import rasterio
    for f in sorted((RAW / "map_friction").glob("*.tif")):
        with rasterio.open(f) as r: a = r.read(1, masked=True); print(f.name, r.shape, r.res, "min/km median:", float(np.ma.median(a)) * 1000)

## 11. Population rasters

In [ ]:
import rasterio
with rasterio.open(RAW / "worldpop/phl_ppp_2020_UNadj.tif") as r:
    a = r.read(1, out_shape=(r.height // 10, r.width // 10), masked=True)  # decimated preview
    full = sum(np.nansum(np.where(blk == r.nodata, 0, blk)) for _, w in r.block_windows(1) for blk in [r.read(1, window=w)])
    print("WorldPop 2020 UN-adj:", r.shape, r.res, f"total = {full/1e6:.1f} M")
plt.figure(figsize=(5, 6)); plt.imshow(np.log1p(a), cmap="magma"); plt.axis("off"); plt.title("WorldPop 2020 (log, decimated)"); plt.show()
for z in sorted((RAW / "meta_hrsl").glob("*.zip")):
    tif = [n for n in zipfile.ZipFile(z).namelist() if n.endswith(".tif")][0]
    with rasterio.open(f"zip://{z.as_posix()}!{tif}") as r:
        tot = sum(np.nansum(blk) for _, w in r.block_windows(1) for blk in [r.read(1, window=w)]); print(z.name, r.shape, f"total = {tot/1e6:.2f} M")
note("worldpop", f"WorldPop 2020 UN-adjusted total {full/1e6:.1f} M (PSA 2020 CPH: 109.0 M).")

## Summary: what the data say about the proposal

In [ ]:
pd.DataFrame(FINDINGS, columns=["dataset", "finding"]).style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})